# [심화·선택] 08. Multi-agent / Sub-agent (Advanced) — 제품 탑재형 AI: Fleet Telemetry 분석 시나리오
**Main Agent(감독) ↔ Data Analyst(Python) ↔ Writer ↔ QA Reviewer**

---

## 🧩 시나리오(가상)
스마트 생활가전/로보틱스 제품이 대량으로 배포되었을 때, 아래 같은 분석을 수행합니다.

- 특정 모델에서 **에러코드/고장률**이 급증하는지
- 지역/펌웨어 버전별로 **이상 패턴**이 있는지
- 고객지원/현장 기사 대응을 위해 **근거 기반 리포트**를 빠르게 작성해야 하는지

이 노트북은 더미 텔레메트리 데이터를 만들고,
**LLM 멀티에이전트로 (1)분석 → (2)리포트 작성 → (3)QA 검수** 흐름을 실습합니다.

> **사전 조건 (로컬 LLM)**: `03_llama_server_local_llm.ipynb`에서 띄운 **llama-server(http://localhost:8080 · 기본 Qwen3.5-4B / 옵션 Qwen2.5-7B)** 가 실행 중이어야 합니다.
> 서버가 꺼져 있다면 터미널에서 `python local_server.py start 4b` 를 실행하세요. (API 키/비용 없음, 모든 추론은 내 GPU에서 수행)
> 로컬 환경에서는 GPU가 하나이므로, 역할별 에이전트가 **같은 로컬 서버를 temperature만 다르게** 호출합니다 (multi-agent single-LLM 구조).



이 노트북은 6시간 핵심 과정 이후 진행하는 심화 선택 자료입니다.


이 노트북은 6시간 핵심 과정 이후 진행하는 심화 선택 자료입니다.


## 🧭 시작 전에

| 항목 | 내용 |
|---|---|
| 🎯 학습 목표 | 역할이 다른 여러 LLM 에이전트(분석가·작성자·검수자)를 연결해 데이터 분석 보고서를 자동으로 만든다 |
| 📚 사전 지식 | 05번(구조화 출력), 06번(에이전트), pandas 기초 |
| 🐍 커널(환경) | `LLM Hands-On (Win CUDA)` |
| 🖥️ GPU 사용 | 공용 llama-server |
| ⏱️ 소요 시간 | 약 50분 (파이프라인 1회 실행에 수 분) |
| ✔️ 실행 전 확인 | llama-server 실행 중 |

> **실행 요령**: 위에서부터 **한 셀씩**(Shift+Enter) 실행하세요. `Run All`은 쓰지 마세요.
> `✅ TODO 셀`은 학생이 채우는 칸이라 그대로 실행하면 일부러 에러가 납니다. 막히면 바로 아래 `✅ 참고 답안` 셀을 보세요.

## 📖 핵심 개념: 왜 에이전트를 여러 개로 나눌까?

한 에이전트에게 "분석하고, 보고서 쓰고, 검수까지 해"라고 하면 프롬프트가 길고 복잡해져 **작은 모델일수록 실수가 늘어납니다.**
사람 팀처럼 **역할을 나누면** 각 프롬프트가 짧고 명확해집니다.

```
질문 ─▶ [Data Analyst] ─(JSON 수치)─▶ [Report Writer] ─(보고서)─▶ [QA Reviewer] ─(검수 결과)─▶ FinalReport(JSON)
          LLM이 pandas 코드 작성            숫자로 문장 작성          숫자·논리 점검
          → Python이 실제 계산
```

| 포인트 | 설명 |
|---|---|
| **LLM은 계산을 못한다** | LLM은 숫자를 "그럴듯하게" 지어낼 수 있음 → 분석가는 **코드를 작성**하고, 계산은 **Python이 실제로 실행** |
| **Supervisor(감독관) 패턴** | 감독 에이전트가 하위 에이전트를 도구처럼 호출 (`run_pipeline`) |
| **고정 파이프라인** | 순서를 코드로 고정 (`run_pipeline_single_pass`) → 작은 로컬 모델에서 더 안정적 |
| **로컬 GPU 1장** | 역할별 에이전트가 같은 서버를 **순서대로** 호출 (temperature만 다름) |

> ⚠️ LLM이 만든 코드를 `exec`로 실행합니다. 실습용 더미 데이터에서만 사용하고, 실서비스에서는 격리된 환경(샌드박스)이 필요합니다.

In [ ]:
# 맨 처음 한 번만 실행하세요. (패키지 없으면 create_agent 등 import 실패)
# !pip -q install -U langchain langchain-openai langchain-core langgraph pandas numpy pydantic requests


In [ ]:
import os, json, io, contextlib
import pandas as pd
import numpy as np

from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from typing import Literal, Optional, List, Dict, Any

# ── 로컬 LLM 연결 (공용 설정: local_llm.py) ──
# 모든 앱 실습(05~09)은 같은 로컬 OpenAI 호환 서버(llama-server)를 사용합니다.
from local_llm import get_chat_model, check_server, LLM_BASE_URL, LLM_MODEL
check_server()   # 서버가 안 떠 있으면 해결 방법을 알려주고 에러를 냅니다.


## 1) 데이터 생성

- 기간: 2025-01 ~ 2025-03 (일 단위)
- 지역: Seoul, Busan, Daejeon
- 기기 타입: robot_vacuum / washing_machine / air_conditioner
- 펌웨어 버전: 기기 타입별 2~3개
- 지표: power_w, bump_events(로봇), error_code(간헐 발생)

> 이 데이터는 Sub-agent(Data Analyst)의 Python 분석 대상입니다.


In [ ]:
rng = np.random.default_rng(7)

# 더미 텔레메트리 데이터 생성 (실데이터 아님)
days = pd.date_range("2025-01-01", "2025-03-31", freq="D")
regions = ["Seoul", "Busan", "Daejeon"]
device_types = ["robot_vacuum", "washing_machine", "air_conditioner"]
fw_versions = {
    "robot_vacuum": ["1.1.0", "1.2.0", "1.2.1"],
    "washing_machine": ["2.1.5", "2.1.7"],
    "air_conditioner": ["3.5.0", "3.5.1", "3.5.2"],
}

rows = []
for d in days:
    for r in regions:
        for dt in device_types:
            # 하루/지역/기기타입별 80~160대 샘플(임의)
            n = int(rng.integers(80, 161))
            for _ in range(n):
                fw = rng.choice(fw_versions[dt])
                # 에러율(임의): 특정 버전에서 washer 5C 증가하도록 의도적으로 bias
                base_err = 0.01
                if dt == "washing_machine" and fw == "2.1.7" and d >= pd.Timestamp("2025-02-10"):
                    base_err = 0.05
                # robot vacuum은 bump 이벤트가 많은 날(주말) 증가시키는 편향
                bump = int(rng.poisson(6 if d.day_of_week >= 5 else 3))   # dayofweek는 pandas 신버전에서 제거 예정
                err_code = None
                if rng.random() < base_err:
                    err_code = "5C" if dt == "washing_machine" else ("E2" if dt == "air_conditioner" else "BUMP_STUCK")

                power_w = float(max(50, rng.normal(200 if dt=="robot_vacuum" else 600 if dt=="air_conditioner" else 400, 80)))
                rows.append({
                    "date": d.strftime("%Y-%m-%d"),
                    "region": r,
                    "device_type": dt,
                    "fw": fw,
                    "power_w": round(power_w, 1),
                    "bump_events": bump if dt=="robot_vacuum" else 0,
                    "error_code": err_code,
                })

df = pd.DataFrame(rows)
df.head()


### 🔍 정답지 만들기: LLM 보고서를 검증할 "진짜 수치"

LLM 결과를 믿기 전에 **직접 계산한 정답**을 준비해 둡시다. 나중에 보고서의 숫자와 비교해 환각이 있는지 확인합니다.
(데이터는 2025-02-10 이후 펌웨어 2.1.7 세탁기의 5C 에러율이 높아지도록 일부러 만들었습니다)

In [ ]:
wm = df[df["device_type"] == "washing_machine"].copy()
wm["period"] = np.where(wm["date"] >= "2025-02-10", "after_0210", "before_0210")
truth = (wm.assign(is_5c=wm["error_code"].eq("5C"))
           .groupby(["fw", "period"])["is_5c"].mean().unstack().round(4))
print("세탁기 5C 에러율 (정답지)")
truth

**실행 순서**: 반드시 **위쪽 "1) 데이터 생성" 셀을 먼저 실행**한 뒤 이 셀을 실행하세요.  
`df`가 없으면 `LocalPythonREPL({"df": df, ...})`에서 `NameError`가 납니다.

## 2) Sub-agents 정의

### (A) Data Analyst
- df를 사용해 수치 계산, 필터링, 이상치 탐지

### (B) Report Writer
- Data Analyst 결과(숫자/표)를 받아 자연어 보고서 작성

### (C) QA Reviewer
- 숫자/논리/누락을 체크하고 개선 지시를 작성


In [ ]:
# LLM (역할별로 온도 다르게)

# single-agent multi-tool
# agent1 = create_agent(model=..., tools=[A, B, C], ...)

# multi-agent single-LLM
# agent1 = create_agent(model=llmX, tools=[A], ...)
# agent2 = create_agent(model=llmX, tools=[B, C], ...)

# multi-agent hetero-LLM
# agent1 = create_agent(model=llmX, tools=[A], ...)
# agent2 = create_agent(model=llmY, tools=[B, C], ...)
# -> tool을 다루는 능력이 모델마다 큰 차이가 있을 때, 특화를 원하면 해볼 수 있는 구현
# -> 단, llmX와 llmY가 로컬 모델이라고 하면 GPU가 병목


# 로컬: 하나의 llama-server(4B/7B 프리셋)를 역할별 temperature로 호출
planner_llm = get_chat_model(temperature=0)
writer_llm  = get_chat_model(temperature=0.3)
qa_llm      = get_chat_model(temperature=0)

# Data Analyst는 Python 실행기 포함
# (langchain-experimental의 PythonAstREPLTool 대체 — 추가 패키지/버전 충돌 없이 동일하게 동작)
class LocalPythonREPL:
    """df/pd/np가 들어있는 네임스페이스에서 코드를 실행하고 print 출력(stdout)을 반환합니다.
    ⚠ LLM이 만든 코드를 그대로 실행하므로 실습(더미 데이터) 용도로만 사용하세요."""
    def __init__(self, namespace: dict):
        self.ns = dict(namespace)

    def run(self, code: str) -> str:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            exec(code, self.ns)          # 예외는 호출부(data_analyst_agent_fn)에서 JSON 에러로 변환
        return buf.getvalue()

python_tool = LocalPythonREPL({"df": df, "pd": pd, "np": np})

# 소형 로컬 모델이 컬럼명을 지어내지 않도록 df 스키마를 프롬프트에 함께 제공
DF_SCHEMA = (
    f"df 행 수: {len(df)}\n"
    f"df 컬럼/타입: {dict(df.dtypes.astype(str))}\n"
    "date는 'YYYY-MM-DD' 문자열, error_code는 대부분 None(에러 없음)이고 에러 시 '5C'/'E2'/'BUMP_STUCK'\n"
    f"예시 행: {df.head(3).to_dict(orient='records')}\n"
)

# create_agent 루프 대신: LLM이 코드 생성 → Python 직접 실행 (재귀 없음)
def data_analyst_agent_fn(query: str) -> str:
    """LLM으로 pandas 코드 생성 후 python_tool로 실행. 결과를 JSON 문자열로 반환."""
    code_prompt = (
        "df, pd, np 가 이미 import/정의되어 있습니다.\n"
        f"{DF_SCHEMA}"
        "아래 분석 요청에 맞는 pandas/numpy 코드를 작성해, 마지막 줄에 반드시 print(json.dumps(result, ensure_ascii=False, default=str))로 결과를 출력하세요.\n"
        "result는 dict이고 metrics, notes, evidence 키를 포함해야 합니다.\n"
        "코드만 출력하세요. 설명·마크다운 금지.\n\n"
        f"[분석 요청]\n{query}"
    )
    resp = planner_llm.invoke(code_prompt)
    code = resp.content.strip()
    if code.startswith("```"):
        lines = code.split("\n")
        code = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
    code = "import json\n" + code
    try:
        result_txt = python_tool.run(code)
    except Exception as e:
        result_txt = json.dumps({"metrics": {}, "notes": f"코드 실행 오류: {e}", "evidence": code[:300]})
    return result_txt or json.dumps({"metrics": {}, "notes": "결과 없음", "evidence": code[:300]})

report_writer_agent = create_agent(
    model=writer_llm,
    tools=[],
    system_prompt=(
        "당신은 제품 텔레메트리 리포트 라이터입니다.\n"
        "- 분석가의 JSON 결과를 읽고, 한국어 보고서로 재구성하세요.\n"
        "- 숫자 근거를 반드시 포함하고, 과장된 단정은 피하세요.\n"
        "- 마지막에 '추가로 확인하면 좋은 데이터'를 2개 제안하세요."
    )
)

qa_reviewer_agent = create_agent(
    model=qa_llm,
    tools=[],
    system_prompt=(
        "당신은 QA 리뷰어입니다.\n"
        "- 보고서에서 숫자/논리/누락/모호한 표현을 찾으세요.\n"
        "- 문제가 있으면 '수정 지시'를 구체적으로 작성하세요.\n"
        "- 문제가 없으면 OK만 반환하세요."
    )
)


## 3) Sub-agent들을 Tool로 래핑

Main Agent는 아래 도구들만 호출할 수 있고, **직접 계산하지 않습니다.**


In [ ]:
def _extract_json_from_text(s: str) -> str:
    if not s or not s.strip(): return s
    s = s.strip()
    start = s.find("{")
    if start == -1: return s
    return s[start: s.rfind("}") + 1]

@tool
def ask_data_analyst(query: str) -> str:
    """데이터 분석이 필요할 때 호출합니다. df 기반 분석을 수행하고 JSON 문자열을 반환합니다."""
    print("[1/3] Data Analyst 실행 중...")
    txt = data_analyst_agent_fn(query)
    print("[1/3] Data Analyst 완료.")
    return _extract_json_from_text(txt or "")

@tool
def write_report(analysis_json: str) -> str:
    """분석가 결과(JSON 문자열)를 받아 보고서를 작성합니다."""
    print("[2/3] Report Writer 실행 중...")
    res = report_writer_agent.invoke({"messages": [{"role": "user", "content": analysis_json}]})
    print("[2/3] Report Writer 완료.")
    return res["messages"][-1].content

@tool
def qa_review(report_text: str) -> str:
    """보고서를 QA 검수하고 수정 지시 또는 OK를 반환합니다."""
    print("[3/3] QA Reviewer 실행 중...")
    res = qa_reviewer_agent.invoke({"messages": [{"role": "user", "content": report_text}]})
    print("[3/3] QA Reviewer 완료.")
    return res["messages"][-1].content


## 4) 최종 보고서 스키마 (구조화 출력)

Main Agent의 최종 출력은 아래 스키마를 따라야 합니다.


In [ ]:
class KeyMetricEntry(BaseModel):
    """지표 이름과 값 (API strict schema용)"""
    key: str
    value: str

class FinalReport(BaseModel):
    short_answer: str
    key_metrics: List[KeyMetricEntry]
    narrative: str
    qa_result: str
    next_questions: List[str]



## 5) Main Agent(감독관) 구성

규칙:
- 질문을 읽고 필요한 분석을 `ask_data_analyst`에 위임
- 결과를 `write_report`로 보고서화
- `qa_review`로 검수
- 최종 결과를 `FinalReport`로 구조화하여 반환

> 업그레이드 포인트:  
> - 감독관은 직접 계산하지 않고, **분석/작성/검수**를 모두 위임해야 합니다.


> **두 가지 실행 방식 비교**
> - `run_pipeline(q)`: 감독관 LLM이 도구 호출 순서를 스스로 결정 → 유연하지만, 작은 모델은 같은 도구를 여러 번 부르거나 순서를 틀릴 수 있음
> - `run_pipeline_single_pass(q)`: 순서를 코드로 고정 → **기본값으로 사용**. 진행 단계가 `[1/3]…[3/3]`으로 출력됩니다.

In [ ]:
main_agent = create_agent(
    model=planner_llm,
    tools=[ask_data_analyst, write_report, qa_review],
    system_prompt=(
        "당신은 감독관(Main Agent)입니다.\n"
        "반드시 다음 순서로 각 도구를 **정확히 한 번씩만** 호출하세요.\n"
        "1) ask_data_analyst(질문) 호출 → 2) write_report(분석 JSON) 호출 → 3) qa_review(보고서 원문) 호출\n"
        "세 번 호출이 끝나면 더 이상 도구를 호출하지 말고, 그 결과를 바탕으로 최종 출력만 FinalReport 스키마로 제출하세요."
    ),
    response_format=ToolStrategy(FinalReport),
)

def run_pipeline(question: str) -> FinalReport:
    res = main_agent.invoke({"messages": [{"role": "user", "content": question}]})
    return res["structured_response"]

# 도구를 정확히 한 번씩만 호출하는 고정 파이프라인 (반복 호출 방지)
def run_pipeline_single_pass(question: str) -> FinalReport:
    analysis = ask_data_analyst.invoke({"query": question})
    report_text = write_report.invoke({"analysis_json": analysis})
    qa_result = qa_review.invoke({"report_text": report_text})
    out = planner_llm.with_structured_output(FinalReport).invoke(
        f"""다음 분석/보고서/QA 결과를 FinalReport 스키마로 정리하세요. key_metrics는 key, value 문자열 쌍 리스트로 넣으세요.\n\n분석(JSON): {analysis}\n\n보고서: {report_text}\n\nQA 결과: {qa_result}"""
    )
    return out


In [ ]:
# 예시 (분석 → 보고서 → QA → 구조화)
q = "2025-02-10 이후 세탁기(washing_machine)에서 5C 에러가 증가하는지 확인하고, 특정 펌웨어 버전과의 상관을 근거(지표)와 함께 설명해줘."
print("파이프라인 시작 (진행 단계가 출력됩니다)...")
# Main Agent 사용 (가끔 write_report/qa_review를 여러 번 호출할 수 있음)
# report = run_pipeline(q)
# 각 도구를 정확히 한 번씩만 쓰는 고정 파이프라인
report = run_pipeline_single_pass(q)
report


## 📝 실습 과제

### 문제 1) 에러 복구 루프 추가 (상)
Data Analyst가 가끔 JSON이 아닌 텍스트를 반환하거나, 코드 실행이 실패할 수 있습니다.

- `ask_data_analyst`에서 반환값을 `json.loads()`로 파싱해보고,
  실패하면 **재시도 프롬프트**를 1회 더 보내서 JSON 형태로 고치게 하세요.

### 문제 2) 이상 패턴 탐지 기준 강화 (상)
- 단순히 "가장 큰 값"을 찾는 대신,
  - 특정 날짜 이후 **에러율이 3배 이상 증가**
  - 또는 최근 7일 평균 대비 **+2σ 이상 스파이크**
  같은 기준으로 이상 패턴을 탐지하도록 Data Analyst 시스템 프롬프트를 개선하세요.

### 문제 3) 보고서 품질 QA를 구조화 (중~상)
- QA 결과를 단순 OK 문자열이 아니라,
  - issues(list)
  - severity(enum)
  - fix(list)
  로 구조화해서 FinalReport에 넣도록 확장해보세요.

> 아래 셀은 학생용 TODO입니다.  
> (강사용 답안은 바로 아래에 예시로 제공합니다.)


In [ ]:
# ✅ TODO 셀 (학생용)
# - ask_data_analyst를 JSON 파싱 + 1회 재시도로 업그레이드
# - Data Analyst 시스템 프롬프트 개선

raise NotImplementedError("학생용 TODO 셀입니다. 아래 '참고 답안'을 보기 전까지 구현해보세요.")


In [ ]:
# ✅ 참고 답안 (핵심만) — 막힐 때만 참고하세요

# --- 문제 1) JSON 파싱 + 1회 재시도 ---
@tool
def ask_data_analyst_v2(query: str) -> str:
    """df 분석 후 JSON 반환. 파싱 실패 시 LLM 에게 1회 재시도 요청."""
    txt = _extract_json_from_text(data_analyst_agent_fn(query) or "")
    try:
        json.loads(txt)
        return txt
    except Exception:
        resp = planner_llm.invoke(
            f"아래 텍스트에서 metrics/notes/evidence 키를 가진 순수 JSON만 출력하세요.\n\n{txt[:800]}"
        )
        txt2 = _extract_json_from_text(resp.content or "")
        try:
            json.loads(txt2)
            return txt2
        except Exception:
            return json.dumps({"metrics": {}, "notes": "JSON 변환 실패", "evidence": (txt or txt2)[:500]})


# --- 문제 2) 이상 패턴 탐지 기준 강화 (Data Analyst 프롬프트에 주입) ---
ANOMALY_CRITERIA = """
[이상 패턴 탐지 기준 — 두 기준 모두 계산해 결과에 포함]
1. [3배 기준] 기준일/특정 fw 전후 에러율 비교: after_rate / before_rate >= 3.0 이면 이상.
   evidence 에 before_rate, after_rate, ratio 명시.
2. [2σ 기준] 일별 에러율의 최근 7일 평균이 (전체 평균 + 2*표준편차)를 넘으면 스파이크.
   evidence 에 global_mean, global_std, recent_mean, threshold 명시.
해당 기준을 evidence["anomaly_flags"] 리스트에 기록.
"""
# -> data_analyst_agent_fn 의 code_prompt 안에 f"{ANOMALY_CRITERIA}" 를 추가하면 됩니다.


# --- 문제 3) QA 결과 구조화 (issues / severity / fix) ---
class QAIssueReport(BaseModel):
    issues:   List[str]
    severity: Literal["critical", "warning", "ok"]
    fix:      List[str]

class FinalReport(BaseModel):
    short_answer:   str
    key_metrics:    List[KeyMetricEntry]
    narrative:      str
    qa_result:      QAIssueReport      # str → 구조화 스키마로 변경
    next_questions: List[str]
# -> qa_reviewer_agent 시스템 프롬프트가 위 JSON(issues/severity/fix)만 출력하도록 바꾸고,
#    qa_review 툴에서 json.loads 후 setdefault 로 누락 키를 보정해 반환하면 됩니다.

## ✅ 정리 & 스스로 점검

**정리**
- 복잡한 일은 **역할별 에이전트**로 나누면 작은 모델로도 품질이 올라갑니다.
- 숫자는 LLM이 아니라 **코드가 계산**하게 하고, LLM은 코드 작성과 문장화에 집중시킵니다.
- 흐름 제어를 LLM에 맡길지(감독관), 코드로 고정할지(파이프라인)는 안정성과 유연성의 트레이드오프입니다.

**스스로 점검**
1. 보고서의 에러율 수치가 위의 **정답지(truth)** 와 일치했나요? 다르다면 어느 단계에서 틀렸을까요?
2. QA Reviewer가 "OK"만 반환했다면 정말 문제가 없는 걸까요? (문제 3의 구조화 QA)
3. 같은 GPU 1장에서 에이전트 3개를 동시에(병렬로) 돌리면 빨라질까요?

**다음 단계 →** 09번: 지금까지 배운 것들을 **그래프(LangGraph)** 로 묶는 종합 실습입니다.